# Decision Trees and Pruning

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

Decision trees classify by recursively splitting predictor space into regions with mostly one class.

**Highest-probability exam pattern:** Fit a classification tree, tune cost-complexity pruning, and report test or CV accuracy.

## Assumptions, intuition, and theory

Unpruned trees can overfit because they keep splitting until leaves are too specific. Cost-complexity pruning trades training fit for simpler trees and better generalization.

## Python method notes and documentation

`DecisionTreeClassifier` fits recursive splits, `cost_complexity_pruning_path` gives candidate pruning penalties, `ccp_alpha` controls pruning, and `plot_tree` visualizes the fitted rules.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [plot_tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for pruning results.
import matplotlib.pyplot as plt  # Import plotting tools.
from sklearn.datasets import make_moons  # Import nonlinear classification simulator.
from sklearn.metrics import accuracy_score  # Import classification accuracy.
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # Import splitting and CV tools.
from sklearn.tree import DecisionTreeClassifier, plot_tree  # Import tree classifier and plotting helper.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_moon_classification_data(n=220, noise=0.30, random_state=123):  # Define a nonlinear classification simulator.
    X, y = make_moons(  # Generate two interlocking classes.
        n_samples=n,  # Set the number of observations.
        noise=noise,  # Add overlap and noise.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the simulator call.
    return X, y  # Return predictors and labels.


In [ ]:
X, y = make_moon_classification_data(n=220, noise=0.30, random_state=RANDOM_SEED)  # Simulate data with nonlinear class boundaries.
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=RANDOM_SEED)  # Create a stratified train/test split.
base_tree = DecisionTreeClassifier(random_state=RANDOM_SEED)  # Create an unpruned decision tree.
path = base_tree.cost_complexity_pruning_path(X_train, y_train)  # Compute candidate pruning penalties from the training data.
alpha_values = path.ccp_alphas[:-1]  # Drop the final alpha that prunes the tree to only the root.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)  # Define stratified CV for pruning selection.
rows = []  # Create a list for pruning results.
for alpha in alpha_values[:20]:  # Limit candidates for a fast exam-style tuning loop.
    tree = DecisionTreeClassifier(random_state=RANDOM_SEED, ccp_alpha=alpha)  # Create a tree with this pruning penalty.
    scores = cross_val_score(tree, X_train, y_train, cv=cv, scoring='accuracy')  # Estimate training-fold validation accuracy.
    rows.append({'ccp_alpha': alpha, 'mean_cv_accuracy': np.mean(scores)})  # Store the candidate result.
results = pd.DataFrame(rows)  # Convert pruning results to a table.
best_alpha = float(results.loc[results['mean_cv_accuracy'].idxmax(), 'ccp_alpha'])  # Select alpha with highest CV accuracy.
pruned_tree = DecisionTreeClassifier(random_state=RANDOM_SEED, ccp_alpha=best_alpha)  # Create the selected pruned tree.
pruned_tree.fit(X_train, y_train)  # Fit the pruned tree on the training data.
test_accuracy = accuracy_score(y_test, pruned_tree.predict(X_test))  # Evaluate the pruned tree on held-out data.
display(results.head(10))  # Display the first pruning candidates.
print(f'Best ccp_alpha: {best_alpha:.6f}')  # Print selected pruning penalty.
print(f'Test accuracy after pruning: {test_accuracy:.3f}')  # Print held-out accuracy.
plt.figure(figsize=(8, 4))  # Create a compact tree plot figure.
plot_tree(pruned_tree, max_depth=3, filled=True, feature_names=['x1', 'x2'], class_names=['0', '1'])  # Plot the top of the pruned tree.
plt.title('Pruned decision tree')  # Title the tree plot.
plt.show()  # Render the tree plot.


## Exam-style problem prompt

A prompt asks for a decision-tree classifier and pruning. Use `cost_complexity_pruning_path`, tune `ccp_alpha` by CV, and evaluate the selected tree.

## Adaptable code pattern

```python
# Step 1: Split data into train and test sets.
# Step 2: Fit an initial tree and get pruning alphas.
# Step 3: Use CV accuracy to choose ccp_alpha.
# Step 4: Refit the pruned tree on the training data.
# Step 5: Evaluate test accuracy and optionally plot the tree.
```

## What to remember for the exam

Trees are easy to explain but overfit easily. Pruning is the exam-friendly way to show you understand the bias-variance tradeoff.